In [1]:
import pandas as pd
import os

def process_and_save_datasets(directory_path):
    # Get all CSV files in the directory
    files = [f for f in os.listdir(directory_path) if f.endswith('.csv')]
    
    # Create a directory to save the output files if it doesn't exist
    output_dir = os.path.join(directory_path, 'output_data')
    os.makedirs(output_dir, exist_ok=True)
    
    # Iterate through each file
    for file in files:
        file_path = os.path.join(directory_path, file)
        df = pd.read_csv(file_path)
        
        if 'datetime' not in df.columns:
            print(f"'datetime' column missing in {file}, skipping this file.")
            continue

        # Convert the datetime column to datetime format
        df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')

        if df['datetime'].isnull().all():
            print(f"Failed to parse 'datetime' column in {file}, skipping this file.")
            continue

        # Filter data for the year 2023
        df = df[df['datetime'].dt.year == 2023]

        # If no data remains after filtering, skip the file
        if df.empty:
            print(f"No data for the year 2023 in {file}, skipping this file.")
            continue

        # Extracting the day of the month, day of the week, and hour of the day
        df['day_of_month'] = df['datetime'].dt.day
        df['day_of_week'] = df['datetime'].dt.day_name()  # Use day names
        df['hour_of_day'] = df['datetime'].dt.hour
        df['month'] = df['datetime'].dt.month_name()

        # Group by the extracted intervals and count the occurrences
        day_of_month_counts = df.groupby('day_of_month')['Event'].count().reset_index()
        day_of_week_counts = df.groupby('day_of_week')['Event'].count().reindex(
            ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']).reset_index()
        hour_of_day_counts = df.groupby('hour_of_day')['Event'].count().reset_index()

        # Save counts to CSV files
        day_of_month_counts.to_csv(os.path.join(output_dir, f'{file}_day_of_month_counts.csv'), index=False)
        day_of_week_counts.to_csv(os.path.join(output_dir, f'{file}_day_of_week_counts.csv'), index=False)
        hour_of_day_counts.to_csv(os.path.join(output_dir, f'{file}_hour_of_day_counts.csv'), index=False)

        # Save counts for each month separately
        for month, month_data in df.groupby('month'):
            day_of_month_counts_by_month = month_data.groupby('day_of_month')['Event'].count().reset_index()
            day_of_month_counts_by_month.to_csv(os.path.join(output_dir, f'{file}_day_of_month_counts_{month}.csv'), index=False)

    print(f'All data have been saved to {output_dir}')

# Example usage:
directory_path = 'E:\Economic_Data\Output data'
process_and_save_datasets(directory_path)


No data for the year 2023 in Fed_Chair_Yellen_Speaks.csv, skipping this file.
All data have been saved to E:\Economic_Data\Output data\output_data
